In [6]:
!git clone https://github.com/kapilverse/SLM.git
%cd SLM

Cloning into 'SLM'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 44 (delta 13), reused 44 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 16.65 KiB | 294.00 KiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/SLM/SLM


In [7]:
!pip install -q tiktoken gradio datasets

In [15]:
%cd /content/SLM
!git pull
!python prepare_tinystories.py --output data/tinystories.txt --num-stories 500000

/content/SLM
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 6 (delta 4), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 838 bytes | 838.00 KiB/s, done.
From https://github.com/kapilverse/SLM
   4dcf057..7d46931  main       -> origin/main
Updating 4dcf057..7d46931
Fast-forward
 checkpoint.py          | 5 ++++-
 prepare_tinystories.py | 3 +++
 2 files changed, 7 insertions(+), 1 deletion(-)
Writing 500000 stories to data/tinystories.txt ...
Done.


In [16]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True Tesla T4


In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
from config import TrainConfig
from train import train

train_cfg = TrainConfig(checkpoint_dir="/content/drive/MyDrive/SLM_checkpoints")
model, tok = train(data_path="data/tinystories.txt", train_cfg=train_cfg)

Using device: cuda
step    250 | train_loss 5.3787 | val_loss 5.3868 | 36.0s
step    500 | train_loss 4.5296 | val_loss 4.6428 | 60.2s
Saved checkpoint at step 500 -> /content/drive/MyDrive/SLM_checkpoints/ckpt_step500.pt
step    750 | train_loss 4.3110 | val_loss 4.2946 | 86.4s
step   1000 | train_loss 4.0876 | val_loss 4.0313 | 111.6s
Saved checkpoint at step 1000 -> /content/drive/MyDrive/SLM_checkpoints/ckpt_step1000.pt
step   1250 | train_loss 4.0464 | val_loss 3.8594 | 139.1s
step   1500 | train_loss 3.7230 | val_loss 3.7440 | 168.0s
Saved checkpoint at step 1500 -> /content/drive/MyDrive/SLM_checkpoints/ckpt_step1500.pt
step   1750 | train_loss 3.6678 | val_loss 3.6364 | 195.8s
step   2000 | train_loss 3.5478 | val_loss 3.5524 | 223.1s
Saved checkpoint at step 2000 -> /content/drive/MyDrive/SLM_checkpoints/ckpt_step2000.pt
step   2250 | train_loss 3.6730 | val_loss 3.5135 | 251.6s
step   2500 | train_loss 3.5425 | val_loss 3.4302 | 278.7s
Saved checkpoint at step 2500 -> /conten

In [19]:
from config import TrainConfig
from train import train

train_cfg = TrainConfig(
    checkpoint_dir="/content/drive/MyDrive/SLM_checkpoints",
    max_steps=30000,          # total steps target, not additional
    checkpoint_interval=1000, # save less often over a longer run
)
model, tok = train(
    data_path="data/tinystories.txt",
    train_cfg=train_cfg,
    resume_from="/content/drive/MyDrive/SLM_checkpoints/ckpt_step5000.pt",
)

Using device: cuda
Resumed from step 5000
step   5250 | train_loss 3.2020 | val_loss 3.0355 | 41.5s
step   5500 | train_loss 3.0891 | val_loss 3.0174 | 68.8s
step   5750 | train_loss 3.0379 | val_loss 2.9968 | 95.7s
step   6000 | train_loss 3.1529 | val_loss 2.9739 | 124.6s
Saved checkpoint at step 6000 -> /content/drive/MyDrive/SLM_checkpoints/ckpt_step6000.pt
step   6250 | train_loss 3.2372 | val_loss 2.9595 | 152.3s
step   6500 | train_loss 3.0572 | val_loss 2.9296 | 179.7s
step   6750 | train_loss 3.2258 | val_loss 2.9105 | 207.7s
step   7000 | train_loss 3.0108 | val_loss 2.8925 | 234.9s
Saved checkpoint at step 7000 -> /content/drive/MyDrive/SLM_checkpoints/ckpt_step7000.pt
step   7250 | train_loss 3.0996 | val_loss 2.8918 | 263.9s
step   7500 | train_loss 3.1252 | val_loss 2.8624 | 291.6s
step   7750 | train_loss 2.9815 | val_loss 2.8351 | 318.9s
step   8000 | train_loss 2.9517 | val_loss 2.8269 | 346.0s
Saved checkpoint at step 8000 -> /content/drive/MyDrive/SLM_checkpoints/ckp

In [20]:
import torch
from config import GPTConfig, TrainConfig
from model import SmallGPT
from checkpoint import load_checkpoint
from dataset import build_dataloaders, load_text_file
from evaluate import perplexity
from generate import generate
from tokenizer import Tokenizer

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = GPTConfig()
train_cfg = TrainConfig()

In [22]:
model = SmallGPT(cfg).to(device)
load_checkpoint("/content/drive/MyDrive/SLM_checkpoints/ckpt_step30000.pt", model, map_location=device)
model.eval()

SmallGPT(
  (embeddings): GPTEmbeddings(
    (token_emb): Embedding(50257, 128)
    (pos_emb): Embedding(128, 128)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (blocks): ModuleList(
    (0-3): 4 x TransformerBlock(
      (ln1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (attn): GroupedQueryAttention(
        (Wq): Linear(in_features=128, out_features=128, bias=False)
        (Wk): Linear(in_features=128, out_features=64, bias=False)
        (Wv): Linear(in_features=128, out_features=64, bias=False)
        (out_proj): Linear(in_features=128, out_features=128, bias=False)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (ffn): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=128, out_features=512, bias=True)
          (1): ReLU()
          (2): Linear(in_features=512, out_features=128, bias=True)
    

In [23]:
# rebuild the same train/val split used during training
text = load_text_file("data/tinystories.txt")
train_loader, val_loader, tok = build_dataloaders(text, cfg.context_size, train_cfg.batch_size)

In [24]:
ppl = perplexity(model, val_loader, device, max_batches=200)
print(f"Validation perplexity: {ppl:.2f}")

Validation perplexity: 9.60


In [25]:
# 2. Qualitative check: generate from a few prompts
for prompt in ["Once upon a time", "The little dog", "One day, a girl named"]:
    out = generate(model, tok, prompt, max_new_tokens=100, temperature=0.8, top_k=40, device=device)
    print("---")
    print(out)

---
Once upon a time, there was a little girl named Lily. She had a big, soft, fluffy pillow that she loved to sleep safe.

One day, Lily's mommy told her to be careful by mistake. The pillow was dark and dark. Lily's mommy felt scared and scared. 

Lily's mommy saw the trumpet and wanted to stop her from. She ran to the living room and started to cry. Her mommy said, "Don't worry, Lily. We
---
The little dog went up to the fence. The boy was so proud of himself and thanked the dog. From that day on, the little dog made sure to always let him feel better.
<|endoftext|>
Once upon a time, there was a small, blackboard. The walls were very special and could fit in it. One day, the little boy wanted to explore the world. 

The old man said, "No, I want to go!" He climbed up the desk and looked around. But
---
One day, a girl named Amy was feeling very happy and didn't like to play outside. She wanted to find her friends so she asked her friends to play a game. The friends said yes! They wa